In [ ]:
import sqlite3
from pprint import pprint

import pandas as pd

from xer_persona.scraper.config import DB_NAME, TABLE_NAME
from xer_persona.scraper.crawler import run_scraper
from xer_persona.scraper.db import setup_db

In [2]:
# 1. Garante que o banco de dados e a tabela existam
setup_db()

# 2. Executa o processo de raspagem
# Conecta ao banco de dados para passar a conexão para o scraper
try:
    conn = sqlite3.connect(DB_NAME)
    # O `limit_pages=10` é para manter o comportamento original do seu script.
    # Remova ou ajuste conforme necessário.
    run_scraper(conn) 
finally:
    if conn:
        conn.close()
        print("Conexão com o banco de dados fechada.")



Banco de dados '/home/gusarti/pessoal/code/xer_persona/data/contos.sqlite' e tabela 'tales' configurados.
Buscando links de categorias em: https://sites.pitt.edu/~dash/folktexts.html
Encontrados 216 links de categorias.
Raspando página: https://sites.pitt.edu/~dash/type0156.html
  Encontrados 6 contos.
  Aguardando 2.85 segundos...
Raspando página: https://sites.pitt.edu/~dash/grimm200.html
  Encontrados 2 contos.
  Aguardando 2.10 segundos...
Raspando página: https://sites.pitt.edu/~dash/fairytheft.html
  Encontrados 8 contos.
  Aguardando 1.55 segundos...
Raspando página: https://sites.pitt.edu/~dash/type0154.html
  Encontrados 3 contos.
  Aguardando 2.94 segundos...
Raspando página: https://sites.pitt.edu/~dash/mbuilder.html
  Encontrados 14 contos.
  Aguardando 1.47 segundos...
Raspando página: https://sites.pitt.edu/~dash/type0850.html
  Encontrados 16 contos.
  Aguardando 2.68 segundos...
Raspando página: https://sites.pitt.edu/~dash/dragonslayers.html
  Encontrados 0 contos.
  A

In [4]:
# 3. Análise dos dados com Pandas
print("\n--- Análise dos Dados ---")
try:
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
    
    print("Amostra dos 5 primeiros contos:")
    display(df.head())

    print("\nInformações sobre a tabela:")
    df.info()

    print("\nContagem de contos por origem:")
    display(df['origem'].value_counts())

finally:
    if conn:
        conn.close()


--- Análise dos Dados ---
Amostra dos 5 primeiros contos:


,id,titulo,origem,url,texto_completo
0,1,Androcles,Aesop,https://sites.pitt.edu/~dash/type0156.html,But shortly afterwards both Androcles and the ...
1,2,The Slave and the Lion,Aesop,https://sites.pitt.edu/~dash/type0156.html,"A slave ran away from his master, by whom he h..."
2,3,Androcles and the Lion,Joseph Jacobs,https://sites.pitt.edu/~dash/type0156.html,It happened in the old days at Rome that a sla...
3,4,The Lion and the Saint,Andrew Lang,https://sites.pitt.edu/~dash/type0156.html,"The old man with the beard is St. Jerome, who ..."
4,5,Of the Remembrance of Benefits,N/A,https://sites.pitt.edu/~dash/type0156.html,There was a knight who devoted much of his tim...



Informações sobre a tabela:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1665 entries, 0 to 1664
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              1665 non-null   int64 
 1   titulo          1665 non-null   object
 2   origem          1665 non-null   object
 3   url             1665 non-null   object
 4   texto_completo  1665 non-null   object
dtypes: int64(1), object(4)
memory usage: 65.2+ KB

Contagem de contos por origem:


origem
N/A                   282
Germany               149
England                93
Ireland                73
Scotland               54
                     ... 
Poggio Bracciolini      1
Tagalog                 1
Mandaya (Mindanao)      1
Bagobo (Mindanao)       1
Lancashire              1
Name: count, Length: 414, dtype: int64

In [4]:
print("\n--- Buscando um Conto Específico ---")
try:
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    titulo_procurado = 'The Magic Violin'
    
    # Usando f-string para o nome da tabela e '?' para o valor para evitar SQL Injection
    query = f"SELECT texto_completo FROM {TABLE_NAME} WHERE titulo = ?"
    
    cursor.execute(query, (titulo_procurado,))
    
    resultado = cursor.fetchone()
    
    if resultado:
        print(f"Texto completo para '{titulo_procurado}':\n")
        pprint(resultado[0])
    else:
        print(f"O conto com o título '{titulo_procurado}' não foi encontrado.")

finally:
    if conn:
        conn.close()


--- Buscando um Conto Específico ---
Texto completo para 'The Magic Violin':

('A dirty, ragged old man was sitting there. He called to him, "Give me '
 'something, for God\'s sake!""I only have three hellers, but I\'ll give them '
 'to you. In three years I\'ll earn them again. Just take them.""I thank you '
 'very much, and I shall grant you three wishes. What do you choose.""I ask '
 'for a gun that never misses its target, for a violin that makes everyone '
 'dance, and further I wish that no one will be able refuse my requests."The '
 'old man made the wishes come true, and Johann continued on his way, half '
 'dancing, half walking. He came to a forest and stayed there to rest.Suddenly '
 'he heard someone say, "Oh, I\'d gladly give anything if I could have the '
 'beautiful nightingale singing over there on that tree."It was the farmer who '
 'had given Johann three hellers as wages who uttered these words. Johann took '
 'his gun, which never missed its target, and shot down t